# 05 — Il test che conta: impatto sulla protezione (vs proxy)

Hit ratio a parte, un segnale di regime vale se **protegge nei crash**. Overlay di
*market timing*: investito in BULL, cash in BEAR (decisione su info del giorno
prima, niente lookahead). Confronto Calmar / MaxDD: proxy vs segnali-vol vs buy&hold,
su 2002–2026 (include 2008, 2020, 2022).

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import vol_regime as vr
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

In [2]:
def stats(ret):
    ret = ret.dropna(); nav = (1+ret).cumprod(); n = len(ret)
    cagr = nav.iloc[-1]**(252/n) - 1
    mdd = -(nav/nav.cummax() - 1).min()
    sharpe = ret.mean()/ret.std()*np.sqrt(252) if ret.std() else 0
    return dict(CAGR=cagr, Sharpe=sharpe, MaxDD=mdd, Calmar=cagr/mdd if mdd>0 else np.nan)

vol = vr.load_vol(); prices = vr.load_prices()
for ac, vcol in vr.VOL_FOR_CLASS.items():
    px = prices[vr.BENCHMARKS[ac]].dropna(); r = px.pct_change(); v = vol[vcol].dropna()
    start, end = px.index.min(), px.index.max()
    regimes = {
        "proxy(200d)": vr.proxy_regime(ac, start, end),
        f"{vcol} thr q80": vr.signal_threshold(v, q=0.80),
        f"{vcol} mom q80": vr.signal_vol_momentum(v, 21, q=0.80),
        f"{vcol} ts 21/126": vr.signal_term_structure(v, 21, 126),
    }
    rows = {"buy&hold": stats(r)}
    for name, reg in regimes.items():
        rg = reg.reindex(px.index).ffill().shift(1)
        rows[name] = stats(r.where(rg == vr.BULL, 0.0))
    print(f"==== {ac} ({vcol}-timed, {start.date()}→{end.date()}) ====")
    print(pd.DataFrame(rows).T[["CAGR","Sharpe","MaxDD","Calmar"]].to_string(float_format=lambda x: f"{x:.3f}"))
    print()

==== Equity (VIX-timed, 2000-01-03→2026-06-26) ====
               CAGR  Sharpe  MaxDD  Calmar
buy&hold      0.063   0.412  0.565   0.111
proxy(200d)   0.054   0.544  0.210   0.258
VIX thr q80   0.019   0.215  0.614   0.032
VIX mom q80   0.052   0.426  0.457   0.114
VIX ts 21/126 0.024   0.264  0.550   0.044

==== Fixed Income (MOVE-timed, 2002-07-30→2026-06-26) ====
                 CAGR  Sharpe  MaxDD  Calmar
buy&hold        0.003   0.092  0.518   0.006
proxy(200d)    -0.006  -0.004  0.393  -0.015
MOVE thr q80   -0.002   0.041  0.419  -0.004
MOVE mom q80    0.018   0.210  0.495   0.037
MOVE ts 21/126 -0.017  -0.131  0.462  -0.038



## Conclusione onesta (livello C)

**I segnali-vol gratuiti NON battono il proxy.**

- **Equity**: il proxy (200d MA) domina — Calmar ~0.26 e MaxDD ~0.21, contro
  Calmar ≤0.11 e MaxDD ≥0.46 dei segnali VIX. La soglia VIX fa *peggio* del
  buy&hold (entra in BEAR a crash iniziato, poi resta fuori nel rimbalzo).
- **Fixed Income**: tutto debole (MOVE non aiuta; TLT stesso rende poco).

**Decisione** (paletti della spec): nessun segnale passa A+B+C → **non si scrive
`OptionsRegimeProvider`**, non si promuove nulla. Il proxy resta il giudice e,
per ora, il migliore.

**Prossimo bivio (per la chat di progetto):**
1. tenere il proxy e concentrarsi sui **segnali ML del motore** (trend/oscillatori/SVM); oppure
2. salire a **Bloomberg** (term structure dei futures VIX, skew/risk-reversal,
   scomposizione IV) — l'unica via per replicare davvero l'approccio AlgoEagle,
   ma è il pezzo costoso: da fare solo se si decide che la pista vale l'investimento.